# Getting to know the Data

Before you start coding or planning your strategy it is best to dive into the data that you will be working with.

**What is a 10-K?** A 10-K is the comprehensive annual report that public companies in the U.S. are required to file with the Securities and Exchange Commission (SEC). It gives a detailed picture of the company's business: what it does, the risks it faces, its financial results, and management's discussion of those results. The report is organized into standardized sections called *items* (Item 1 – Business, Item 1A – Risk Factors, Item 7 – Management's Discussion & Analysis, and so on), which is exactly the `item_1`, `item_1A`, `item_7` ... structure you'll see in these files.

**Where the data comes from.** The original, unprocessed filings are public. You can browse them for any company through the SEC's full-text search at [https://www.sec.gov/edgar/search/](https://www.sec.gov/edgar/search/) — handy if you ever want to sanity-check a file or see the raw formatting.

**What we provide.** You don't need to download anything yourself: we have already downloaded the filings, preprocessed them into clean per-item text, and provided them as `.json` files in `TO_ADD`. The rest of this notebook shows how to load one of those files and split a section into paragraphs.


# Splitting a 10-K Filing into Paragraphs

A small template showing how to load one of the 10-K `.json` files and split a
section (an "item") into paragraphs.

**What these files look like.** Each `.json` file is a *flat* dictionary. It has
some metadata fields (`company`, `cik`, `filing_date`, ...) and then the actual
filing text, broken into sections named `item_1`, `item_1A`, `item_2`, ...,
`item_16`. Every item is just a single string, and inside that string the
paragraphs are separated by newline characters (`\n`).

So the whole task is really just: pick an item string and `.split("\n")` it.


## 1. Load a file

Point `FILE_PATH` at whichever filing you want to inspect.

In [ ]:
import json
from pathlib import Path

FILE_PATH = Path("1800_10K_2019_0001104659-20-023904.json") # TODO when on Git

with open(FILE_PATH, encoding="utf-8") as f:
    filing = json.load(f)

# `filing` is a plain dict. Let's see the metadata and which items exist.
metadata_keys = [k for k in filing if not k.startswith("item_")]
item_keys     = [k for k in filing if k.startswith("item_")]

print("Company:", filing["company"])
print("Filing type / date:", filing["filing_type"], "/", filing["filing_date"])
print()
print("Metadata fields:", metadata_keys)
print()
print("Item sections:", item_keys)

## 2. Look at one raw section

Before splitting, it helps to see what the raw text looks like. Each item is one
long string with `\n` between paragraphs.

In [ ]:
item_1_text = filing["item_1"]

print("Type:", type(item_1_text).__name__)
print("Character length:", len(item_1_text))
print()
print("First 500 characters (note the \\n line breaks):")
print(repr(item_1_text[:500]))

## 3. Split into paragraphs

The whole thing is one line: `text.split("\n")`. These filings use a single
newline between paragraphs and contain no blank lines, so a plain split is all
that's needed. We just `.strip()` each paragraph to trim any stray spaces.

The function below is reusable for any item (`item_1A`, `item_7`, ...).

In [ ]:
def split_into_paragraphs(text):
    """Split a filing-item string into a list of paragraphs.

    Parameters
    ----------
    text : str
        The raw item text (e.g. filing["item_1"]).

    Returns
    -------
    list[str]
        The paragraphs, in order.
    """
    return [p.strip() for p in text.split("\n")]


paragraphs = split_into_paragraphs(item_1_text)
print(f"Item 1 contains {len(paragraphs)} paragraphs.")

## 4. Print the first and the fifth paragraph

Remember Python is zero-indexed: the first paragraph is `paragraphs[0]` and the
fifth is `paragraphs[4]`.

In [ ]:
def show(label, paragraph):
    print(label)
    print("-" * len(label))
    print(paragraph)
    print()

show("First paragraph (paragraphs[0]):", paragraphs[0])
show("Fifth paragraph (paragraphs[4]):", paragraphs[4])

## 5. Putting it together

A tiny helper that goes straight from a file path + item name to a paragraph
list, so you can reuse this on any of the five files.

In [ ]:
def load_item_paragraphs(file_path, item="item_1"):
    with open(file_path, encoding="utf-8") as f:
        data = json.load(f)
    return split_into_paragraphs(data[item])


# Example: try it on a different file / item
example = load_item_paragraphs(FILE_PATH, "item_1")
print(f"{len(example)} paragraphs")
print("First paragraph:", example[0])